# SET50 Data Visualization

This notebook loads prepared SET50 CSV files safely and plots price/volume without scale distortion.

In [ ]:
# Run this once if imports fail, then restart the kernel.
# %pip install pandas matplotlib seaborn

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (14, 6)

DATA_DIR = Path("data-prepared")
FILES = {
    "Daily": DATA_DIR / "SET50_days.csv",
    "Weekly": DATA_DIR / "SET50_weeks.csv",
    "Monthly": DATA_DIR / "SET50_months.csv",
}

NUMERIC_COLS = ["Close", "Open", "High", "Low", "Volume", "Change_pct", "Target_Next_Close"]

def load_set50_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, thousands=",")
    df.columns = df.columns.str.strip()
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col].astype(str).str.replace(",", "", regex=False).str.replace("%", "", regex=False),
                errors="coerce",
            )

    df = df.dropna(subset=["Date", "Close"]).sort_values("Date").reset_index(drop=True)
    return df

dfs = {name: load_set50_csv(path) for name, path in FILES.items()}
df_day = dfs["Daily"]
df_week = dfs["Weekly"]
df_month = dfs["Monthly"]

for name, df in dfs.items():
    print(f"{name}: {len(df):,} rows | {df['Date'].min().date()} to {df['Date'].max().date()}")
    print(df[NUMERIC_COLS].dtypes)
    print()

df_day.head()


In [ ]:
# Plot Close separately by timeframe.
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=False)

for ax, (name, df) in zip(axes, dfs.items()):
    ax.plot(df["Date"], df["Close"], linewidth=1.6, label="Close")
    ax.set_title(f"SET50 {name} Close")
    ax.set_ylabel("Index")
    ax.legend(loc="upper left")

plt.tight_layout()
plt.show()


In [ ]:
# Overlay close prices. Monthly/weekly have fewer points, so they look smoother.
plt.figure(figsize=(14, 6))
plt.plot(df_day["Date"], df_day["Close"], label="Daily", alpha=0.35, linewidth=1)
plt.plot(df_week["Date"], df_week["Close"], label="Weekly", alpha=0.85, linewidth=1.5)
plt.plot(df_month["Date"], df_month["Close"], label="Monthly", linewidth=2.2)
plt.title("SET50 Close by Timeframe")
plt.xlabel("Date")
plt.ylabel("Index")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Close + Volume needs two y-axes because Volume is billions while Close is around hundreds/thousands.
for name, df in dfs.items():
    fig, ax1 = plt.subplots(figsize=(14, 5))
    ax2 = ax1.twinx()

    ax1.plot(df["Date"], df["Close"], color="#2563eb", linewidth=1.6, label="Close")
    ax2.bar(df["Date"], df["Volume"] / 1_000_000_000, color="#94a3b8", alpha=0.35, label="Volume (B)")

    ax1.set_title(f"SET50 {name}: Close and Volume")
    ax1.set_ylabel("Close index", color="#2563eb")
    ax2.set_ylabel("Volume (B)", color="#475569")

    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc="upper left")

    plt.tight_layout()
    plt.show()


In [ ]:
# Direction label for prediction: 1 = next close is higher, 0 = down or flat.
direction_parts = []

for name, df in dfs.items():
    tmp = df.dropna(subset=["Target_Next_Close"]).copy()
    tmp["Direction"] = (tmp["Target_Next_Close"] > tmp["Close"]).map({True: "Up", False: "Down/Flat"})
    direction_parts.append(tmp.assign(Timeframe=name))

direction_df = pd.concat(direction_parts, ignore_index=True)

print(pd.crosstab(direction_df["Timeframe"], direction_df["Direction"], normalize="index").mul(100).round(2))

plt.figure(figsize=(8, 4))
sns.countplot(data=direction_df, x="Timeframe", hue="Direction", order=list(dfs.keys()))
plt.title("Target Direction Count")
plt.xlabel("")
plt.ylabel("Rows")
plt.tight_layout()
plt.show()
